# **Vision-Based Sense and Avoid**

# FIX ME!!!!!!

## **Monocular Airborne Object Tracking for Safe Drone Flight**

This notebook provides an end-to-end demonstration of the final trained model for this project, covering dataset download, model weight loading, and result production on the custom version of the Airborne Object Tracking (AOT) dataset. The AOT dataset is a real-world collection of 5.9M+ images across diverse flight conditions, where airborne objects can appear as just a few pixels per frame, making this one of the most challenging small-object detection problems in computer vision, and one of the most critical for future autonomous flight systems.

*For full project setup, dataset details, and source scripts see the [Project README](../README.md).*

In [ ]:
# Import Required Libraries.
import os
import sys

# Set Working Directory to Project Root.
os.chdir("..") 

# Add Project Root to Path.
sys.path.insert(0, os.getcwd())


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets** - Airborne objects occupying only a few pixels per frame.
- **Multiple Targets** - Scenes containing more than one airborne object simultaneously.
- **Poor Visibility** - Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Import Data Downloader Libraries.
from src.data.download_data import main as downloadData

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "data/hard_dataset_flights.txt"

# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)


---

<br/>

## **Baseline Model Evaluation**

This section evaluates the winning solution from the AICrowd Airborne Object Tracking Challenge (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is run across all three parts of the hard dataset subset, and results are aggregated into a single set of benchmark statistics using the official AICrowd evaluation metrics.

**Evaluation Metrics:**
- **AFDR** — Airborne Flight Detection Rate. The primary detection metric.
- **FPPI** — False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR** — Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR** — Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ **Note:** Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.


---

<br/>